# 09 – Expression Evaluator

ducto uses a **safe AST-based** evaluator — not `eval()`. It whitelists
specific node types and function names. Dangerous operations
(`__import__`, `open`, `globals`) are blocked.

Available variables: `input_tokens`, `output_tokens`, `cache_read_tokens`,
`cache_write_tokens`, `tool_calls`, `search_queries`, `search_results`,
`web_search_calls`, `code_exec_calls`.

Available functions: `percentile(base, pct)`, `abs`, `round`, `min`, `max`, `sum`.

### Basic arithmetic

In [ ]:
from ducto.expr import evaluate_expression

r = evaluate_expression("input_tokens * 5 + output_tokens * 15",
                        {"input_tokens": 500, "output_tokens": 200})
print(f"  500×5 + 200×15 = {r}  (expected: 5500)")
assert r == 5500

### Function calls

In [ ]:
r = evaluate_expression("max(input_tokens, output_tokens) * 2",
                        {"input_tokens": 500, "output_tokens": 200})
print(f"  max(500,200)×2 = {r}  (expected: 1000)")
assert r == 1000

### Percentile function

`percentile(p, v1, v2, ...)` computes the `p`-th percentile of the values.
For example, `percentile(90, 100, 200, 300)` finds the value below which
90 % of the data falls.

In [ ]:
r = evaluate_expression("percentile(input_tokens, 100, 200, 300)",
                        {"input_tokens": 90})
print(f"  percentile(input_tokens, 100, 200, 300) = {r}")
# 90th percentile of (100, 200, 300) = 280 (linear interpolation)

### Combined expression

In [ ]:
expr = "input_tokens * 3 + output_tokens * 15 + max(tool_calls, 0) * 10"
r = evaluate_expression(expr, {"input_tokens": 1000, "output_tokens": 400, "tool_calls": 1})
print(f"  {r}")
# = 3000 + 6000 + 1*10 = 9010

### Safety — blocked operations

These are **blocked** by the AST whitelist. Uncomment to test:

In [ ]:
# evaluate_expression("__import__('os').system('ls')", {})
# evaluate_expression("open('/etc/passwd').read()", {})
# evaluate_expression("globals()", {})
# evaluate_expression("lambda x: x", {})
print("All dangerous operations blocked by AST whitelist.")